# **Bike Rentals Demand Forecasting: Modeling**

![rainbow](https://github.com/ancilcleetus/My-Learning-Journey/assets/25684256/839c3524-2a1d-4779-85a0-83c562e1e5e5)

# 📖 TABLE OF CONTENTS

- [Imports](#Imports)
- [Feature Description](#Feature-Description)
- [Baseline Model](#Baseline-Model)
  - [Target Definition](#Target-Definition)
- [References](#References)

![rainbow](https://github.com/ancilcleetus/My-Learning-Journey/assets/25684256/839c3524-2a1d-4779-85a0-83c562e1e5e5)

# Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor
import matplotlib.pyplot as plt

from typing import Union, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

![rainbow](https://github.com/ancilcleetus/My-Learning-Journey/assets/25684256/839c3524-2a1d-4779-85a0-83c562e1e5e5)

# Feature Description

In [2]:
# Loading the data
data = pd.read_parquet('../data/01_raw/bike_data_train.parquet')
data.head(3)

,datetime,season,hr,holiday,weekday,weathersit,temp,hum,windspeed,cnt
0,2011-01-01 00:00:00,1,0,0,6,1,0.24,0.81,0.0,16
1,2011-01-01 01:00:00,1,1,0,6,1,0.22,0.80,0.0,40
2,2011-01-01 02:00:00,1,2,0,6,1,0.22,0.80,0.0,32


**Column Descriptions**

* **datetime** – Timestamp (date + hour)
* **season** – Season of the year: `1` = Winter, `2` = Spring, `3` = Summer, `4` = Fall
* **hr** – Hour of the day (0–23)
* **holiday** – Whether the day is a public holiday (`0` = No, `1` = Yes)
* **weekday** – Day of the week (`0` = Sunday, `6` = Saturday)
* **weathersit** – Weather condition:
  * `1` = Clear or partly cloudy
  * `2` = Misty or cloudy
  * `3` = Light rain or snow
  * `4` = Heavy rain or snow
* **temp** – Normalized temperature in Celsius
* **hum** – Normalized relative humidity
* **windspeed** – Normalized wind speed
* **cnt** – Total bike rentals (target variable)

In [3]:
# Drop 'datetime' column
drop_cols = ['datetime']
data = data.drop(columns=drop_cols)
data.head(3)

,season,hr,holiday,weekday,weathersit,temp,hum,windspeed,cnt
0,1,0,0,6,1,0.24,0.81,0.0,16
1,1,1,0,6,1,0.22,0.80,0.0,40
2,1,2,0,6,1,0.22,0.80,0.0,32


![rainbow](https://github.com/ancilcleetus/My-Learning-Journey/assets/25684256/839c3524-2a1d-4779-85a0-83c562e1e5e5)

# Baseline Model

We will use CatBoost with raw features as the initial model.

Then, we will add lag features and try other algorithms and check for improvements.

For the metrics, we will use
* Mean Absolute Error (MAE)
* Mean Squared Error (MSE)
* Mean Absolute Percentage Error (MAPE)

In [4]:
def compute_metrics(
    y_true: Union[np.ndarray, list],
    y_pred: Union[np.ndarray, list]
) -> Dict[str, float]:
    """
    Compute evaluation metrics between true and predicted values.

    Metrics returned:
    - MAE: Mean Absolute Error
    - RMSE: Root Mean Squared Error
    - MAPE: Mean Absolute Percentage Error (in %)

    Parameters:
    ----------
    y_true : array-like
        Ground truth values.
    y_pred : array-like
        Predicted values.

    Returns:
    -------
    dict
        Dictionary with keys 'MAE', 'MAPE', and 'RMSE' and their float values.
    """
    y_true = np.array(y_true).ravel()
    y_pred = np.array(y_pred).ravel()

    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    mape = np.mean(np.abs((y_true - y_pred) / y_true + 1e-8)) * 100

    return {
        'MAE': float(round(mae, 2)),
        'RMSE': float(round(rmse, 2)),
        'MAPE': float(round(mape, 2)),
    }

In [5]:
def prepare_dataset(
    df: pd.DataFrame,
    train_fraction: float = 0.8
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """
    Splits a DataFrame into training and testing sets for features and target.

    Parameters:
    ----------
    df : pd.DataFrame
        The input DataFrame that must contain a 'target' column.
    train_fraction : float, optional (default=0.8)
        The fraction of data to use for training (between 0 and 1).

    Returns:
    -------
    x_train : pd.DataFrame
        Training features.
    x_test : pd.DataFrame
        Testing features.
    y_train : pd.Series
        Training target values.
    y_test : pd.Series
        Testing target values.
    """
    feats = [col for col in df.columns if col != 'target']
    x, y = df[feats], df['target']
    train_size = int(train_fraction * df.shape[0])
    x_train, x_test = x[:train_size], x[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]
    return x_train, x_test, y_train, y_test

## Target Definition

In [6]:
# Work on a copy to preserve the original data
df = data.copy()

In [7]:
# Target = next hour's rental count (shifted by 1); last row gets NaN, so copy the previous value
df['target'] = df['cnt'].shift(-1).ffill()

In [8]:
# Drop the current hour's rental count as it's replaced by target (which is next hour's rental count)
df.drop(columns=['cnt'], inplace=True)

In [9]:
df.head(3)

,season,hr,holiday,weekday,weathersit,temp,hum,windspeed,target
0,1,0,0,6,1,0.24,0.81,0.0,40.0
1,1,1,0,6,1,0.22,0.80,0.0,32.0
2,1,2,0,6,1,0.22,0.80,0.0,13.0


![rainbow](https://github.com/ancilcleetus/My-Learning-Journey/assets/25684256/839c3524-2a1d-4779-85a0-83c562e1e5e5)

# References

- **MLOps Bootcamp by Timur Bikmukhametov**

![rainbow](https://github.com/ancilcleetus/My-Learning-Journey/assets/25684256/839c3524-2a1d-4779-85a0-83c562e1e5e5)